# Lesson 8.4: Parent-Child Retrieval and Small-to-Big Chunking

**Companion notebook for Lesson 8.4**

---

| Section | What you will build |
|---|---|
| 1. The Chunk-Size Dilemma | Measure precision vs. context richness at different granularities |
| 2. HR Policy Corpus | 5 parent sections, 25 child sentences, two-layer store |
| 3. Parent-Child Retrieval | Retrieve small → look up big; deduplicate by parent ID |
| 4. Sentence Window | Simpler variant: matched sentence + N surrounding sentences |
| 5. The Hallucination Trap | Show how child-only context leads to incomplete answers |
| 6. Full Evaluation | Retrieval precision + context completeness across 10 queries |
| 7. Framework Code | LlamaIndex + LangChain patterns |
| 8. Claude API | Real generation comparison: child-only vs. parent context |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`  
**Optional (Section 8):** `anthropic`

In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib
# !pip install anthropic   # optional — Section 8

In [ ]:
%matplotlib inline

import os
import warnings
from collections import defaultdict

os.environ['OMP_NUM_THREADS']         = '1'
os.environ['MKL_NUM_THREADS']         = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

def show_plot():
    plt.tight_layout()
    plt.show()

print('Imports ready.')

---
## 1. The Chunk-Size Dilemma

Every RAG developer hits this wall. You cannot tune chunk size without trading one problem for another:

```
┌──────────────────────────────────────────────────────────────────────────┐
│                      THE CHUNK-SIZE DILEMMA                              │
└──────────────────────────────────────────────────────────────────────────┘

  SMALL CHUNKS (1-2 sentences)                 BIG CHUNKS (sections)
  ─────────────────────────────                ─────────────────────────────
  "The maximum 401(k) contribution             "Section 4.2: Retirement Benefits
   limit for 2024 is $23,000."                  ... 800 words about vesting,
                                                 beneficiaries, employer match,
                                                 IRS rules, catch-up limits...
                                                 The 2024 limit is $23,000.
                                                 ... 300 more words ..."

  ✅ Precise retrieval                          ✅ Rich context for the LLM
  ❌ LLM has no context                        ❌ Fuzzy retrieval (embedding
     — fills in with guesses                      averaged over many topics)

  Insight: WHY does the precise fact get diluted in big chunks?
  An embedding is a single vector that AVERAGES the meaning of the full text.
  A 1000-word section on vesting + limits + IRS rules produces a vector
  pulled in many directions. The precise fact about $23,000 is drowned out.

  Fix: decouple what you RETRIEVE ON from what you GENERATE FROM.
  Retrieve on small chunks. Return the parent chunk to the LLM.
```

In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Loading all-MiniLM-L6-v2...')
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print('Embedder ready.')

# The same fact at three granularities
specific_query = 'What is the 401(k) contribution limit for 2024?'

# Granularity 1: child sentence (one fact)
child_text = 'The maximum 401(k) contribution limit for 2024 is $23,000 per year.'

# Granularity 2: parent paragraph (full context, fact buried inside)
parent_text = (
    'The company 401(k) plan allows employees to contribute pre-tax dollars toward '
    'retirement savings. The maximum contribution limit for 2024 is $23,000 per year. '
    'Employees aged 50 and older may make additional catch-up contributions of up to '
    '$7,500 annually. The company matches 100% of contributions up to 4% of your base '
    'salary. Matching contributions vest on a three-year cliff schedule — employees who '
    'leave before three years forfeit all employer matching funds.'
)

# Granularity 3: full section (even more diluted)
section_text = (
    'Section 4: Employee Benefits Overview. Our benefits package is designed to support '
    'your financial security, health, and work-life balance. ' + parent_text +
    ' See the HR portal for enrollment instructions, plan documents, and beneficiary '
    'designation forms. Questions? Contact benefits@company.com.'
)

q_emb       = embedder.encode(specific_query,  convert_to_tensor=True, show_progress_bar=False)
child_emb   = embedder.encode(child_text,       convert_to_tensor=True, show_progress_bar=False)
parent_emb  = embedder.encode(parent_text,      convert_to_tensor=True, show_progress_bar=False)
section_emb = embedder.encode(section_text,     convert_to_tensor=True, show_progress_bar=False)

sim_child   = float(util.cos_sim(q_emb, child_emb))
sim_parent  = float(util.cos_sim(q_emb, parent_emb))
sim_section = float(util.cos_sim(q_emb, section_emb))

print(f'Query: {specific_query}')
print()
print(f'{"Granularity":<20} {"Text length (chars)":<22} {"Cosine sim to query"}')
print('-' * 62)
print(f'{"Child sentence":<20} {len(child_text):<22} {sim_child:.4f}')
print(f'{"Parent paragraph":<20} {len(parent_text):<22} {sim_parent:.4f}')
print(f'{"Full section":<20} {len(section_text):<22} {sim_section:.4f}')
print()
print('The child sentence has the highest similarity because its embedding')
print('points squarely at the one fact the query is asking about.')
print('Larger chunks produce diluted vectors — the fact competes with surrounding topics.')

In [ ]:
# Scatter plot: precision vs. context richness for different chunk sizes
# Precision = cosine similarity to a specific query
# Context richness = number of supporting facts in the chunk

# Data points (each is a chunk at a different granularity)
points = [
    ('Single sentence\n(child)',     sim_child,   1,   '#E53935'),
    ('Full paragraph\n(parent)',     sim_parent,  5,   '#F57F17'),
    ('Full section\n(large)',        sim_section, 10,  '#888888'),
    ('Parent-child\n(this lesson)',  sim_child,   5,   '#2E7D32'),  # same precision as child + parent context
]

fig, ax = plt.subplots(figsize=(9, 6))

for label, precision, richness, color in points:
    marker = '*' if label.startswith('Parent-child') else 'o'
    size   = 400 if label.startswith('Parent-child') else 180
    ax.scatter(richness, precision, color=color, s=size, zorder=4,
               marker=marker, edgecolors='white', linewidths=0.8)
    ax.annotate(label, (richness, precision),
                textcoords='offset points', xytext=(8, -10),
                fontsize=9, color=color, fontweight='bold')

# Draw the trade-off curve for single-granularity chunking
x_curve = np.linspace(1, 10, 100)
y_curve = 0.42 + (sim_child - 0.42) * np.exp(-0.35 * (x_curve - 1))
ax.plot(x_curve, y_curve, '--', color='#888', linewidth=1.5, alpha=0.6,
        label='Single-granularity trade-off curve')

ax.set_xlabel('Context richness (number of supporting facts in chunk)', fontsize=11)
ax.set_ylabel('Retrieval precision (cosine similarity to query)', fontsize=11)
ax.set_title('The Chunk-Size Dilemma — and How Parent-Child Escapes It',
              fontweight='bold')
ax.set_ylim(0.4, 1.0)

legend_handles = [
    mpatches.Patch(color='#E53935', label='Small chunks: precise, no context'),
    mpatches.Patch(color='#888888', label='Large chunks: rich context, diluted retrieval'),
    mpatches.Patch(color='#2E7D32', label='Parent-child ★: precise retrieval + rich context'),
]
ax.legend(handles=legend_handles, fontsize=9, loc='lower right')
ax.annotate('Parent-child breaks\nthe trade-off curve',
            (5, sim_child), textcoords='offset points', xytext=(30, 30),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2),
            color='#2E7D32', fontsize=10, fontweight='bold')

show_plot()

---
## 2. HR Policy Corpus — Two-Layer Store

5 parent sections (paragraphs), each split into 4-5 child sentences.  
The docstore holds parent texts; the vector store holds child embeddings.

```
┌──────────────────────────────────────────────────────────────────────┐
│               TWO-LAYER STORAGE ARCHITECTURE                         │
└──────────────────────────────────────────────────────────────────────┘

  DOCSTORE (key-value)              VECTOR STORE (child embeddings)
  ─────────────────────             ────────────────────────────────────
  p0 → [401(k) paragraph]           c00 → embed("The company 401(k)...")
  p1 → [health ins. paragraph]      c01 → embed("The maximum limit...")
  p2 → [RSU paragraph]              c02 → embed("Catch-up contributions...")
  p3 → [PTO paragraph]              c03 → embed("Company matches 100%...")
  p4 → [remote work paragraph]      ...  (25 child embeddings total)

  At query time:
    1. Cosine search over child embeddings → top-k child IDs
    2. child.parent_id → docstore lookup → parent texts
    3. Parent texts → LLM (precise retrieval + rich context)
```

In [ ]:
# ── DOCSTORE: parent paragraphs (rich context, never embedded) ────────────────
DOCSTORE = {
    'p0': (
        'The company 401(k) plan allows employees to contribute pre-tax dollars toward '
        'retirement savings. The maximum contribution limit for 2024 is $23,000 per year. '
        'Employees aged 50 and older may make additional catch-up contributions of up to '
        '$7,500 annually. The company matches 100% of contributions up to 4% of your base '
        'salary. Matching contributions vest on a three-year cliff schedule — employees who '
        'leave before three years forfeit all employer matching funds.'
    ),
    'p1': (
        'The company offers three health insurance tiers: Bronze, Silver, and Gold. '
        'Bronze premiums are $120/month for individuals and $340/month for families. '
        'Silver premiums are $180/month individual and $510/month for families, with '
        'lower deductibles. Gold plan premiums are $250/month individual and $710/month '
        'for families, with a $500 annual deductible. Open enrollment occurs each November '
        'for coverage beginning January 1st of the following year.'
    ),
    'p2': (
        'Full-time employees at Level 4 and above receive Restricted Stock Units (RSUs) '
        'as part of their compensation package. RSUs vest over four years with a one-year '
        'cliff: 25% vests after 12 months, then 1/48th of the total grant vests monthly. '
        'RSUs are subject to ordinary income tax upon vesting, not at the grant date. '
        'Employees who leave during the vesting period forfeit unvested RSUs with no '
        'compensation. RSU grants are reviewed annually during the performance review cycle.'
    ),
    'p3': (
        'Full-time employees accrue paid time off (PTO) at a rate depending on years of '
        'service. Employees in their first three years receive 15 days of PTO per year. '
        'After three years of continuous service, PTO increases to 20 days annually. '
        'Employees with eight or more years of service receive 25 days per year. '
        'Unused PTO up to 10 days may be carried over into the following calendar year; '
        'any excess is forfeited on January 1st.'
    ),
    'p4': (
        'The company follows a hybrid work model with a minimum of three in-office days '
        'per week for most roles. Remote work requests exceeding two consecutive weeks '
        'require approval from both the direct manager and HR. Employees relocating to '
        'another country need a formal international remote work agreement, which may '
        'affect compensation and benefits. A one-time home office stipend of $1,000 is '
        'provided to employees transitioning to a hybrid arrangement. Internet '
        'reimbursement of up to $80 per month is available for approved remote workers.'
    ),
}

PARENT_TITLES = {
    'p0': '401(k) Retirement Plan',
    'p1': 'Health Insurance',
    'p2': 'Stock Units (RSU)',
    'p3': 'Paid Time Off (PTO)',
    'p4': 'Remote Work Policy',
}

print('Docstore: 5 parent paragraphs')
for pid, title in PARENT_TITLES.items():
    word_count = len(DOCSTORE[pid].split())
    print(f'  {pid}: {title} ({word_count} words)')

In [ ]:
# ── CHILD CHUNKS: individual sentences with parent pointers ───────────────────
CHILDREN = [
    # 401(k) — parent p0
    {'id': 'c00', 'parent_id': 'p0', 'pos': 0,
     'text': 'The company 401(k) plan allows employees to contribute pre-tax dollars toward retirement savings.'},
    {'id': 'c01', 'parent_id': 'p0', 'pos': 1,
     'text': 'The maximum contribution limit for 2024 is $23,000 per year.'},
    {'id': 'c02', 'parent_id': 'p0', 'pos': 2,
     'text': 'Employees aged 50 and older may make additional catch-up contributions of up to $7,500 annually.'},
    {'id': 'c03', 'parent_id': 'p0', 'pos': 3,
     'text': 'The company matches 100% of contributions up to 4% of your base salary.'},
    {'id': 'c04', 'parent_id': 'p0', 'pos': 4,
     'text': 'Matching contributions vest on a three-year cliff schedule — employees who leave before three years forfeit all employer matching funds.'},

    # Health insurance — parent p1
    {'id': 'c05', 'parent_id': 'p1', 'pos': 0,
     'text': 'The company offers three health insurance tiers: Bronze, Silver, and Gold.'},
    {'id': 'c06', 'parent_id': 'p1', 'pos': 1,
     'text': 'Bronze premiums are $120/month for individuals and $340/month for families.'},
    {'id': 'c07', 'parent_id': 'p1', 'pos': 2,
     'text': 'Silver premiums are $180/month individual and $510/month for families, with lower deductibles.'},
    {'id': 'c08', 'parent_id': 'p1', 'pos': 3,
     'text': 'Gold plan premiums are $250/month individual and $710/month for families, with a $500 annual deductible.'},
    {'id': 'c09', 'parent_id': 'p1', 'pos': 4,
     'text': 'Open enrollment occurs each November for coverage beginning January 1st of the following year.'},

    # RSUs — parent p2
    {'id': 'c10', 'parent_id': 'p2', 'pos': 0,
     'text': 'Full-time employees at Level 4 and above receive Restricted Stock Units (RSUs) as part of their compensation.'},
    {'id': 'c11', 'parent_id': 'p2', 'pos': 1,
     'text': 'RSUs vest over four years with a one-year cliff: 25% vests after 12 months, then 1/48th monthly.'},
    {'id': 'c12', 'parent_id': 'p2', 'pos': 2,
     'text': 'RSUs are subject to ordinary income tax upon vesting, not at the grant date.'},
    {'id': 'c13', 'parent_id': 'p2', 'pos': 3,
     'text': 'Employees who leave during the vesting period forfeit unvested RSUs with no compensation.'},
    {'id': 'c14', 'parent_id': 'p2', 'pos': 4,
     'text': 'RSU grants are reviewed annually during the performance review cycle.'},

    # PTO — parent p3
    {'id': 'c15', 'parent_id': 'p3', 'pos': 0,
     'text': 'Full-time employees accrue paid time off (PTO) at a rate depending on years of service.'},
    {'id': 'c16', 'parent_id': 'p3', 'pos': 1,
     'text': 'Employees in their first three years receive 15 days of PTO per year.'},
    {'id': 'c17', 'parent_id': 'p3', 'pos': 2,
     'text': 'After three years of continuous service, PTO increases to 20 days annually.'},
    {'id': 'c18', 'parent_id': 'p3', 'pos': 3,
     'text': 'Employees with eight or more years of service receive 25 days per year.'},
    {'id': 'c19', 'parent_id': 'p3', 'pos': 4,
     'text': 'Unused PTO up to 10 days may be carried over; any excess is forfeited on January 1st.'},

    # Remote work — parent p4
    {'id': 'c20', 'parent_id': 'p4', 'pos': 0,
     'text': 'The company follows a hybrid model with a minimum of three in-office days per week for most roles.'},
    {'id': 'c21', 'parent_id': 'p4', 'pos': 1,
     'text': 'Remote work requests exceeding two consecutive weeks require approval from both manager and HR.'},
    {'id': 'c22', 'parent_id': 'p4', 'pos': 2,
     'text': 'Employees relocating to another country need a formal international remote work agreement.'},
    {'id': 'c23', 'parent_id': 'p4', 'pos': 3,
     'text': 'A one-time home office stipend of $1,000 is provided to employees transitioning to a hybrid arrangement.'},
    {'id': 'c24', 'parent_id': 'p4', 'pos': 4,
     'text': 'Internet reimbursement of up to $80 per month is available for approved remote workers.'},
]

print(f'Children: {len(CHILDREN)} sentences across {len(DOCSTORE)} parents')
for pid in DOCSTORE:
    kids = [c['id'] for c in CHILDREN if c['parent_id'] == pid]
    print(f'  {pid} ({PARENT_TITLES[pid]}): {kids}')

In [ ]:
# Embed all child chunks — this is the VECTOR STORE
child_texts  = [c['text'] for c in CHILDREN]
child_embeds = embedder.encode(child_texts, convert_to_tensor=True, show_progress_bar=False)

# Also embed parent chunks for the baseline comparison
parent_texts  = [DOCSTORE[pid] for pid in sorted(DOCSTORE.keys())]
parent_ids    = sorted(DOCSTORE.keys())
parent_embeds = embedder.encode(parent_texts, convert_to_tensor=True, show_progress_bar=False)


def baseline_retrieve(query: str, top_k: int = 3):
    """Baseline: embed and search directly over parent paragraphs."""
    q_emb   = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores  = util.cos_sim(q_emb, parent_embeds)[0].cpu().numpy()
    indices = np.argsort(scores)[::-1][:top_k]
    return [(parent_ids[i], float(scores[i])) for i in indices]


print(f'Embedded {len(CHILDREN)} children + {len(DOCSTORE)} parents.')
print()

# Show baseline retrieval on a precise query
demo_q = 'What is the 401(k) contribution limit for 2024?'
print(f'Baseline (parent-level) retrieval for: {demo_q}')
print()
for pid, score in baseline_retrieve(demo_q):
    snippet = DOCSTORE[pid][:80]
    print(f'  {pid} ({PARENT_TITLES[pid]}) score={score:.4f}')
    print(f'     {snippet}...')

print()
print('The baseline returns the right parent (p0), but with a diluted score.')
print('Now watch how child retrieval finds the exact sentence with a much higher score.')

---
## 3. Parent-Child Retrieval

Retrieve with precision (child embeddings) → return with context (parent paragraphs).

In [ ]:
def child_retrieve(query: str, top_k: int = 3):
    """Step 1: semantic search over child sentence embeddings."""
    q_emb   = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores  = util.cos_sim(q_emb, child_embeds)[0].cpu().numpy()
    indices = np.argsort(scores)[::-1][:top_k]
    return [(CHILDREN[i], float(scores[i])) for i in indices]


def parent_child_retrieve(query: str, top_k_children: int = 5, top_k_parents: int = 3):
    """
    Parent-child retrieval:
      1. Retrieve top-k matching child sentences
      2. Collect their parent IDs (deduplicated)
      3. Return parent chunks from the docstore

    Deduplication: when two children share a parent, return the parent only once
    (using the best child score as the parent score — similar to AutoMergingRetriever).
    """
    matched_children = child_retrieve(query, top_k=top_k_children)

    # Collect unique parent IDs, keeping the best child score for each
    best_score_for_parent = {}
    for child, score in matched_children:
        pid = child['parent_id']
        if pid not in best_score_for_parent or score > best_score_for_parent[pid]:
            best_score_for_parent[pid] = score

    # Sort by score, return top_k_parents
    ranked_parents = sorted(
        best_score_for_parent.items(), key=lambda x: x[1], reverse=True
    )[:top_k_parents]

    return [
        {'parent_id': pid, 'parent_text': DOCSTORE[pid],
         'title': PARENT_TITLES[pid], 'child_score': score}
        for pid, score in ranked_parents
    ], matched_children


print('parent_child_retrieve() ready.')

In [ ]:
# Demo: trace through the full retrieval chain
demo_q = 'What is the 401(k) contribution limit for 2024?'

parents, children = parent_child_retrieve(demo_q, top_k_children=5, top_k_parents=2)

print(f'Query: {demo_q}')
print()
print('Step 1 — Top matching child sentences (from vector store):')
for child, score in children:
    pid   = child['parent_id']
    print(f'  [{child["id"]}] score={score:.4f}  parent={pid} ({PARENT_TITLES[pid]})')
    print(f'       {child["text"]}')

print()
print('Step 2 — Parent chunks returned to the LLM (from docstore):')
for p in parents:
    print(f'  [{p["parent_id"]}] {p["title"]}  (child_score={p["child_score"]:.4f})')
    print(f'  {p["parent_text"]}')
    print()

In [ ]:
# Side-by-side: what each strategy returns to the LLM
demo_queries = [
    'What is the 401(k) contribution limit for 2024?',
    'How much does the company match for 401(k)?',
    'When do employer matching contributions vest?',
    'What is the monthly premium for the Gold health plan?',
]

print(f'{"Query":<52} {"Baseline top-1 (parent sim)":<32} {"Child top-1 (child sim)"}')
print('-' * 120)

for q in demo_queries:
    base_results = baseline_retrieve(q, top_k=1)
    base_pid, base_score = base_results[0]

    child_results = child_retrieve(q, top_k=1)
    top_child, child_score = child_results[0]

    q_short = q[:49] + '...' if len(q) > 49 else q
    base_label  = f'{base_pid} ({base_score:.3f})'
    child_label = f'{top_child["id"]} -> {top_child["parent_id"]} ({child_score:.3f})'
    print(f'{q_short:<52} {base_label:<32} {child_label}')

print()
print('Child retrieval consistently scores higher on the precise query,')
print('but the parent-child system still returns the full paragraph context.')

In [ ]:
# Visualise: similarity scores for baseline vs. child retrieval on each query
eval_queries = [
    ('401k limit 2024',       'What is the 401(k) contribution limit for 2024?',       'p0'),
    ('employer match rate',   'How much does the company match for 401(k)?',           'p0'),
    ('vesting schedule',      'When do employer matching contributions vest?',          'p0'),
    ('gold plan premium',     'What is the monthly premium for the Gold health plan?', 'p1'),
    ('enrollment period',     'When is open enrollment for health insurance?',         'p1'),
    ('RSU vesting cliff',     'What is the vesting cliff for stock units?',            'p2'),
    ('pto after 3 years',     'How many PTO days after 3 years of service?',           'p3'),
    ('pto carryover',         'How many PTO days can I carry over each year?',         'p3'),
    ('home office stipend',   'What is the home office setup stipend?',                'p4'),
    ('internet reimbursement','How much is the internet reimbursement for remote?',    'p4'),
]

baseline_scores = []
child_scores    = []

for _, q, expected_pid in eval_queries:
    base_results = baseline_retrieve(q, top_k=5)
    # Find score for the expected parent
    base_score = next((s for pid, s in base_results if pid == expected_pid), 0.0)

    child_results = child_retrieve(q, top_k=5)
    # Find best child score that belongs to the expected parent
    child_score = max((s for c, s in child_results if c['parent_id'] == expected_pid), default=0.0)

    baseline_scores.append(base_score)
    child_scores.append(child_score)

fig, ax = plt.subplots(figsize=(14, 5))
x     = np.arange(len(eval_queries))
width = 0.38

bars_b = ax.bar(x - width/2, baseline_scores, width, color='#E53935', alpha=0.8, label='Baseline (parent embedding)')
bars_c = ax.bar(x + width/2, child_scores,    width, color='#2E7D32', alpha=0.8, label='Child embedding (parent-child retrieval)')

ax.set_xticks(x)
ax.set_xticklabels([label for label, _, _ in eval_queries], rotation=25, ha='right', fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Cosine similarity to the relevant parent')
ax.set_title('Retrieval Precision: Child Embedding vs. Parent Embedding', fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(0.5, color='#F57F17', linewidth=1.5, linestyle='--', alpha=0.7, label='Typical retrieval threshold')

for bar_b, bar_c in zip(bars_b, bars_c):
    for bar, color in [(bar_b, '#E53935'), (bar_c, '#2E7D32')]:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', fontsize=7, color=color)

show_plot()

avg_gain = np.mean(child_scores) - np.mean(baseline_scores)
print(f'Average similarity to relevant parent:')
print(f'  Baseline (parent embedding): {np.mean(baseline_scores):.3f}')
print(f'  Child embedding:             {np.mean(child_scores):.3f}  (+{avg_gain:.3f})')

---
## 4. Sentence Window Variant

Instead of pre-defined parent paragraphs, dynamically expand the matched sentence by including N sentences on each side. Simpler to implement; no need to pre-define chunk boundaries.

```
  Document sentences:  [S1][S2][S3][S4][S5][S6][S7][S8]

  Query matches S4.
  window_size = 2:

  Returned window:     [S2][S3][S4][S5][S6]   (2 before + matched + 2 after)

  The LLM receives the window, not just S4.
```

| | Parent-child | Sentence window |
|---|---|---|
| Parent boundaries | Pre-defined (paragraphs, sections) | Dynamic (N sentences around match) |
| Storage | Docstore required | Metadata on each node |
| Control | Precise (align with structure) | Approximate (may cross boundaries) |
| Best for | Structured documents | Narrative text without clear sections |

In [ ]:
def sentence_window_retrieve(query: str, window_size: int = 2, top_k: int = 3):
    """
    Retrieve top-k matching sentences, expand each to a window of surrounding sentences.

    window_size=2 returns: 2 sentences before + matched + 2 after = 5 total.
    Neighbours are siblings within the same parent section.
    """
    q_emb   = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores  = util.cos_sim(q_emb, child_embeds)[0].cpu().numpy()
    indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in indices:
        child   = CHILDREN[idx]
        score   = float(scores[idx])
        pid     = child['parent_id']

        # Get all siblings from the same parent, sorted by position
        siblings  = sorted([c for c in CHILDREN if c['parent_id'] == pid], key=lambda x: x['pos'])
        pos       = child['pos']
        win_start = max(0, pos - window_size)
        win_end   = min(len(siblings), pos + window_size + 1)
        window    = siblings[win_start:win_end]

        window_text = ' '.join(s['text'] for s in window)
        results.append({
            'matched_child':  child,
            'score':          score,
            'window_text':    window_text,
            'window_ids':     [s['id'] for s in window],
            'parent_id':      pid,
        })

    return results


# Demo
demo_q = 'What is the 401(k) contribution limit for 2024?'
sw_results = sentence_window_retrieve(demo_q, window_size=2, top_k=2)

print(f'Query: {demo_q}')
print()
for i, r in enumerate(sw_results, 1):
    matched = r['matched_child']
    print(f'Match {i}: [{matched["id"]}] score={r["score"]:.4f}')
    print(f'  Matched sentence: {matched["text"]}')
    print(f'  Window ({len(r["window_ids"])} sentences: {r["window_ids"]})')
    print(f'  Window text: {r["window_text"]}')
    print()

In [ ]:
# Visualise: how window size affects context coverage
test_q    = 'What is the 401(k) contribution limit for 2024?'
q_emb_vis = embedder.encode(test_q, convert_to_tensor=False, show_progress_bar=False)

window_sizes  = [0, 1, 2, 3, 4]
context_words = []
window_sims   = []

for ws in window_sizes:
    results = sentence_window_retrieve(test_q, window_size=ws, top_k=1)
    wtext   = results[0]['window_text']
    wemb    = embedder.encode(wtext, convert_to_tensor=False, show_progress_bar=False)
    wwords  = len(wtext.split())
    # Similarity of window to the full parent paragraph (context completeness proxy)
    parent_emb_np = embedder.encode(DOCSTORE['p0'], convert_to_tensor=False, show_progress_bar=False)
    sim_to_parent = float(np.dot(wemb / np.linalg.norm(wemb), parent_emb_np / np.linalg.norm(parent_emb_np)))
    context_words.append(wwords)
    window_sims.append(sim_to_parent)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.bar(window_sizes, context_words, color='#1565C0', alpha=0.7, label='Words in window')
ax2.plot(window_sizes, window_sims, 'o-', color='#2E7D32', linewidth=2.5,
         markersize=8, label='Similarity to full parent (context completeness)')

ax1.set_xlabel('Window size (sentences on each side)')
ax1.set_ylabel('Words returned to LLM', color='#1565C0')
ax2.set_ylabel('Similarity to full parent paragraph', color='#2E7D32')
ax1.set_title('Window Size vs. Context Coverage (window_size=2 is a common default)',
               fontweight='bold')
ax1.set_xticks(window_sizes)
ax1.set_xticklabels([f'window={ws}\n({2*ws+1} sents)' for ws in window_sizes])

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

show_plot()
print('window_size=2 (7 total sentences) is the LlamaIndex default.')
print('Larger windows increase context but add more tokens to the LLM prompt.')

---
## 5. The Hallucination Trap

When the LLM only receives a single matched sentence, it often fills in missing context from its parametric memory — and gets it wrong.

In [ ]:
# Concrete hallucination examples: child-only context vs. parent context

traps = [
    {
        'query':         'What happens to my 401(k) match if I leave after 2 years?',
        'child_text':    'Matching contributions vest on a three-year cliff schedule.',
        'parent_text':   DOCSTORE['p0'],
        'child_answer':  'You may lose some or all matching funds, depending on the vesting schedule.',
        'parent_answer': 'You forfeit all employer matching funds. The three-year cliff means 0% vests before year 3.',
        'hallucination': 'LLM guessed "some or all" — context says it is ALL (cliff, not graded).',
    },
    {
        'query':         'Does the 401(k) have a catch-up option for older employees?',
        'child_text':    'The maximum contribution limit for 2024 is $23,000 per year.',
        'parent_text':   DOCSTORE['p0'],
        'child_answer':  'No specific catch-up information was found; the standard limit is $23,000.',
        'parent_answer': 'Yes — employees aged 50+ can contribute up to $7,500 extra annually on top of the $23,000 limit.',
        'hallucination': 'Child-only context had no catch-up info, so LLM said none exists (false negative).',
    },
    {
        'query':         'Can I work remotely full-time if I relocate abroad?',
        'child_text':    'Employees relocating to another country need a formal international remote work agreement.',
        'parent_text':   DOCSTORE['p4'],
        'child_answer':  'Yes, with a formal agreement. The details depend on the country.',
        'parent_answer': 'You need a formal international remote work agreement, which may affect your compensation and benefits. Additionally, most roles have a 3-day minimum in-office expectation.',
        'hallucination': 'LLM said yes without the salary/benefits caveat — incorrect and risky.',
    },
]

for i, trap in enumerate(traps, 1):
    print(f'=== Example {i} ===')
    print(f'Query:          {trap["query"]}')
    print()
    print(f'Child-only context:')
    print(f'  {trap["child_text"]}')
    print(f'  LLM answer:  {trap["child_answer"]}')
    print(f'  Problem:     {trap["hallucination"]}')
    print()
    print(f'Parent context (what parent-child returns):')
    print(f'  {trap["parent_text"][:160]}...')
    print(f'  LLM answer:  {trap["parent_answer"]}')
    print()

In [ ]:
# Context completeness score:
# Measure how similar the returned context is to the full parent paragraph (gold context)

def context_completeness(returned_text: str, gold_parent_id: str) -> float:
    gold     = DOCSTORE[gold_parent_id]
    ret_emb  = embedder.encode(returned_text,   convert_to_tensor=False, show_progress_bar=False)
    gold_emb = embedder.encode(gold,             convert_to_tensor=False, show_progress_bar=False)
    return float(np.dot(
        ret_emb  / np.linalg.norm(ret_emb),
        gold_emb / np.linalg.norm(gold_emb)
    ))


completeness_queries = [
    ('What is the 401(k) limit for 2024?',           'p0'),
    ('How much does the company match for 401(k)?',  'p0'),
    ('When do RSUs vest?',                           'p2'),
    ('How many PTO days after 3 years?',             'p3'),
    ('What is the home office stipend?',             'p4'),
]

print(f'{"Query":<52} {"Child-only":<14} {"Sent-window":<14} {"Parent-child"}')
print('-' * 100)

child_comps  = []
sw_comps     = []
pc_comps     = []

for q, gold_pid in completeness_queries:
    # Child-only: return just the matched sentence
    top_child = child_retrieve(q, top_k=1)[0][0]
    child_comp = context_completeness(top_child['text'], gold_pid)

    # Sentence window (window=2)
    sw_result = sentence_window_retrieve(q, window_size=2, top_k=1)[0]
    sw_comp   = context_completeness(sw_result['window_text'], gold_pid)

    # Parent-child: return full parent paragraph
    pc_result, _ = parent_child_retrieve(q, top_k_children=3, top_k_parents=1)
    pc_text = pc_result[0]['parent_text'] if pc_result else ''
    pc_comp = context_completeness(pc_text, gold_pid) if pc_text else 0.0

    child_comps.append(child_comp)
    sw_comps.append(sw_comp)
    pc_comps.append(pc_comp)

    q_short = q[:49] + '...' if len(q) > 49 else q
    print(f'{q_short:<52} {child_comp:<14.3f} {sw_comp:<14.3f} {pc_comp:.3f}')

print()
print(f'Average context completeness:')
print(f'  Child-only:     {np.mean(child_comps):.3f}')
print(f'  Sentence window: {np.mean(sw_comps):.3f}')
print(f'  Parent-child:    {np.mean(pc_comps):.3f}')

---
## 6. Full Evaluation: Retrieval Precision + Context Completeness

In [ ]:
EVAL_SET = [
    {'query': 'What is the 401(k) contribution limit for 2024?',        'gold_pid': 'p0', 'gold_cid': 'c01'},
    {'query': 'How much does the company match for 401k contributions?', 'gold_pid': 'p0', 'gold_cid': 'c03'},
    {'query': 'When do employer matching contributions vest?',            'gold_pid': 'p0', 'gold_cid': 'c04'},
    {'query': 'How much is the catch-up contribution limit?',            'gold_pid': 'p0', 'gold_cid': 'c02'},
    {'query': 'What is the monthly premium for the Gold health plan?',   'gold_pid': 'p1', 'gold_cid': 'c08'},
    {'query': 'When is the open enrollment period for health insurance?', 'gold_pid': 'p1', 'gold_cid': 'c09'},
    {'query': 'When do RSUs vest with the one-year cliff?',              'gold_pid': 'p2', 'gold_cid': 'c11'},
    {'query': 'How many PTO days do I get after 3 years of service?',   'gold_pid': 'p3', 'gold_cid': 'c17'},
    {'query': 'How many vacation days can I carry over each year?',      'gold_pid': 'p3', 'gold_cid': 'c19'},
    {'query': 'What is the internet reimbursement for remote workers?',  'gold_pid': 'p4', 'gold_cid': 'c24'},
]


def recall_at_k_parents(results, gold_pid, k=3):
    pids = [r['parent_id'] for r in results[:k]]
    return 1 if gold_pid in pids else 0


def recall_at_k_children(results, gold_cid, k=3):
    cids = [c['id'] for c, _ in results[:k]]
    return 1 if gold_cid in cids else 0


base_precision   = []
child_precision  = []
pc_precision     = []
child_complete   = []
sw_complete      = []
pc_complete      = []

print(f'{"Query":<52} {"Base@3":<8} {"Child@3":<9} {"PC@3":<7} {"CC-child":<10} {"CC-sw":<8} {"CC-pc"}')
print('-' * 108)

for item in EVAL_SET:
    q, gpid, gcid = item['query'], item['gold_pid'], item['gold_cid']

    base_r  = baseline_retrieve(q, top_k=3)
    b_hit   = 1 if gpid in [pid for pid, _ in base_r] else 0

    child_r = child_retrieve(q, top_k=3)
    c_hit   = 1 if gcid in [c['id'] for c, _ in child_r] else 0

    pc_r, _ = parent_child_retrieve(q, top_k_children=5, top_k_parents=3)
    p_hit   = 1 if gpid in [r['parent_id'] for r in pc_r] else 0

    # Context completeness
    top_child_text  = child_r[0][0]['text'] if child_r else ''
    sw_text         = sentence_window_retrieve(q, window_size=2, top_k=1)[0]['window_text']
    pc_text         = pc_r[0]['parent_text'] if pc_r else ''

    cc_child = context_completeness(top_child_text, gpid)
    cc_sw    = context_completeness(sw_text, gpid)
    cc_pc    = context_completeness(pc_text, gpid) if pc_text else 0.0

    base_precision.append(b_hit)
    child_precision.append(c_hit)
    pc_precision.append(p_hit)
    child_complete.append(cc_child)
    sw_complete.append(cc_sw)
    pc_complete.append(cc_pc)

    q_short = q[:49] + '...' if len(q) > 49 else q
    bm = '✅' if b_hit else '❌'
    cm = '✅' if c_hit else '❌'
    pm = '✅' if p_hit else '❌'
    print(f'{q_short:<52} {bm:<8} {cm:<9} {pm:<7} {cc_child:<10.3f} {cc_sw:<8.3f} {cc_pc:.3f}')

print()
print(f'Recall@3:    Baseline={sum(base_precision)}/{len(EVAL_SET)}  Child={sum(child_precision)}/{len(EVAL_SET)}  ParentChild={sum(pc_precision)}/{len(EVAL_SET)}')
print(f'Context CC:  Child={np.mean(child_complete):.3f}  SentWindow={np.mean(sw_complete):.3f}  ParentChild={np.mean(pc_complete):.3f}')

In [ ]:
# Visualise: retrieval precision AND context completeness side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: retrieval precision (recall@3)
strategies_prec  = ['Baseline\n(parent embed)', 'Child embed', 'Parent-child\n(child embed)']
precision_vals   = [
    sum(base_precision)/len(EVAL_SET),
    sum(child_precision)/len(EVAL_SET),
    sum(pc_precision)/len(EVAL_SET),
]
colors_prec = ['#E53935', '#F57F17', '#2E7D32']

bars = axes[0].bar(strategies_prec, precision_vals, color=colors_prec, alpha=0.85, width=0.5)
axes[0].set_ylim(0, 1.15)
axes[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
axes[0].set_title('Retrieval Precision (Recall@3)\nDoes the right chunk appear in top-3?', fontweight='bold')
for bar, val, n in zip(bars, precision_vals, [sum(base_precision), sum(child_precision), sum(pc_precision)]):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{n}/{len(EVAL_SET)}\n({val:.0%})', ha='center', fontsize=11, fontweight='bold')

# Right: context completeness
strategies_cc = ['Child-only\n(matched sentence)', 'Sentence window\n(window=2)', 'Parent-child\n(full paragraph)']
cc_vals       = [np.mean(child_complete), np.mean(sw_complete), np.mean(pc_complete)]
colors_cc     = ['#E53935', '#F57F17', '#2E7D32']

bars2 = axes[1].bar(strategies_cc, cc_vals, color=colors_cc, alpha=0.85, width=0.5)
axes[1].set_ylim(0, 1.15)
axes[1].set_title('Context Completeness\nSimilarity of returned context to full parent paragraph', fontweight='bold')
for bar, val in zip(bars2, cc_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.3f}', ha='center', fontsize=13, fontweight='bold')

show_plot()

print('Key result:')
print('  Parent-child achieves the SAME retrieval precision as child-only embedding')
print('  but returns context as complete as the full parent paragraph.')
print('  It escapes the precision-completeness trade-off entirely.')

---
## 7. Framework Implementations

LlamaIndex and LangChain both ship parent-child retrieval. The code below is for reference — substitute your own document store and vector store.

In [ ]:
# ── LlamaIndex: HierarchicalNodeParser + AutoMergingRetriever ─────────────────
# pip install llama-index llama-index-llms-openai

llamaindex_hierarchical = '''
from llama_index.core.node_parser import HierarchicalNodeParser
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.retrievers import AutoMergingRetriever

# Three levels: 2048-token grandparent / 512-token parent / 128-token leaf
# Leaf nodes are what get embedded; parents and grandparents sit in docstore
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128]
)

nodes = node_parser.get_nodes_from_documents(documents)
leaf_nodes = [n for n in nodes if len(n.child_nodes or []) == 0]

storage_context = StorageContext.from_defaults()
storage_context.docstore.add_documents(nodes)   # store ALL nodes (leaves + parents)

# Index only the leaves — these are what get embedded and searched
index = VectorStoreIndex(leaf_nodes, storage_context=storage_context)

# AutoMergingRetriever: if > 50% of a parent\'s leaves are retrieved,
# replace them with the parent chunk (saves context window, more coherent)
base_retriever = index.as_retriever(similarity_top_k=6)
retriever = AutoMergingRetriever(base_retriever, storage_context)

nodes = retriever.retrieve("What is the 401(k) limit for 2024?")
# Returns parent (or grandparent) chunks — not the raw leaf sentences
'''

print('=== LlamaIndex: HierarchicalNodeParser + AutoMergingRetriever ===')
print(llamaindex_hierarchical)
print('Three-level hierarchy gives the retriever flexibility:')
print('  1 leaf match → return its 512-token parent')
print('  Many leaves from same parent → merge up to 2048-token grandparent')
print('  Two-level would force a binary choice; three levels allow smooth zoom.')

In [ ]:
# ── LlamaIndex: SentenceWindowNodeParser ──────────────────────────────────────
llamaindex_sentwindow = '''
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

# Each node = one sentence, with metadata storing the surrounding window
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,                           # 3 sentences before + 1 matched + 3 after = 7 total
    window_metadata_key="window",            # key where the window text is stored
    original_text_metadata_key="original",  # key where the matched sentence is stored
)

nodes = node_parser.get_nodes_from_documents(documents)
index = VectorStoreIndex(nodes)  # each node = one sentence (precise embeddings)

query_engine = index.as_query_engine(
    similarity_top_k=5,
    node_postprocessors=[
        # Before the LLM sees the results, replace each matched sentence
        # with its full window (the node\'s "window" metadata value)
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ],
)

response = query_engine.query("What is the 401(k) limit for 2024?")
# Retrieval used sentence embeddings (precise)
# LLM received 7-sentence windows (contextual)
'''

print('=== LlamaIndex: SentenceWindowNodeParser + MetadataReplacementPostProcessor ===')
print(llamaindex_sentwindow)

# ── LangChain: ParentDocumentRetriever ────────────────────────────────────────
langchain_parent_doc = '''
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)  # large parents
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=400)   # small children

# vectorstore: holds child embeddings (used for search)
# docstore:    holds parent texts (returned to the LLM)
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

retriever.add_documents(documents)           # builds both stores
results = retriever.invoke("What is the 401k limit for 2024?")
# results = parent chunks, retrieved via child matches
'''

print()
print('=== LangChain: ParentDocumentRetriever ===')
print(langchain_parent_doc)
print('Same two-layer pattern: vectorstore (child search) + docstore (parent lookup).')

---
## 8. Using a Real LLM (Claude API)

Compare real generation quality: child-only context vs. parent-child context, for a question that requires surrounding context to answer correctly.

Install: `pip install anthropic`  
Set key: `export ANTHROPIC_API_KEY=sk-ant-...`

In [ ]:
ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass


GENERATION_PROMPT = '''\
Answer the question using ONLY the provided context.
If the context does not contain enough information, say so explicitly.
Be concise and precise.

Context:
{context}

Question: {query}

Answer:'''


if ANTHROPIC_AVAILABLE:
    client = anthropic.Anthropic()

    def generate(query: str, context: str) -> str:
        resp = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=300,
            messages=[{'role': 'user',
                       'content': GENERATION_PROMPT.format(context=context, query=query)}],
        )
        return resp.content[0].text.strip()

    # Test queries where the child-only context is dangerously incomplete
    test_cases = [
        {
            'query':   'What happens to my 401(k) match if I resign after 2 years?',
            'gold_pid': 'p0',
        },
        {
            'query':   'Can I work remotely full-time if I move to another country?',
            'gold_pid': 'p4',
        },
    ]

    for tc in test_cases:
        q, gpid = tc['query'], tc['gold_pid']

        # Child-only context
        top_child = child_retrieve(q, top_k=1)[0][0]
        child_ctx = top_child['text']

        # Parent-child context
        pc_result, _ = parent_child_retrieve(q, top_k_children=3, top_k_parents=1)
        parent_ctx   = pc_result[0]['parent_text'] if pc_result else ''

        child_ans  = generate(q, child_ctx)
        parent_ans = generate(q, parent_ctx)

        print(f'Query: {q}')
        print()
        print(f'Child-only context:  {child_ctx}')
        print(f'Child-only answer:   {child_ans}')
        print()
        print(f'Parent context:      {parent_ctx[:200]}...')
        print(f'Parent-child answer: {parent_ans}')
        print('=' * 80)
        print()

else:
    print('anthropic not installed or ANTHROPIC_API_KEY not set.')
    print()
    print('To enable this section:')
    print('  pip install anthropic')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('Expected results when you run with Claude:')
    print()
    print('Q: What happens to my 401(k) match if I resign after 2 years?')
    print()
    print('  Child-only context: "Matching contributions vest on a three-year cliff."')
    print('  Child-only answer:  "You may lose some or all matching funds" [INCOMPLETE]')
    print()
    print('  Parent context: [full 401k paragraph]')
    print('  Parent answer:  "You forfeit ALL employer matching — the cliff schedule')
    print('                   means 0% vests before 3 years." [CORRECT, PRECISE]')
    print()
    print('Q: Can I work remotely full-time if I move to another country?')
    print()
    print('  Child-only context: "...need a formal international remote work agreement."')
    print('  Child-only answer:  "Yes, with a formal agreement." [MISLEADING — misses')
    print('                       the compensation impact and 3-day in-office requirement]')
    print()
    print('  Parent context: [full remote work paragraph]')
    print('  Parent answer:  "You need a formal agreement, which may affect your')
    print('                   compensation and benefits. Note the 3-day in-office')
    print('                   minimum for most roles." [CORRECT, COMPLETE]')

---
## Key Takeaways

1. **The chunk-size dilemma is real and unavoidable with a single granularity.** Small chunks embed precisely but starve the LLM; large chunks give rich context but blur retrieval because the embedding averages over too many topics.

2. **Parent-child retrieval escapes the dilemma by decoupling what you search on from what you generate from.** The vector store holds child sentence embeddings (for precision). The docstore holds parent paragraph texts (for context). Same retrieval precision, full context.

3. **The hallucination trap is real.** When the LLM receives only the matched sentence, it fills in missing context from its parametric memory — and often fills it in wrong. Returning the parent paragraph gives the LLM everything it needs and removes the guessing pressure.

4. **Sentence window is the simpler alternative for unstructured text.** No need to pre-define parent chunks — just retrieve the matching sentence and return N sentences on each side. `window_size=2` (5 sentences total) is a reasonable default; tune based on document density.

5. **AutoMergingRetriever (LlamaIndex) handles the deduplication problem.** When two children from the same parent are both retrieved, the naive approach sends the parent twice. AutoMerging detects this and replaces multiple sibling leaves with a single parent chunk.

6. **Align parent boundaries with document structure whenever possible.** Arbitrary character-count cuts produce parents that split mid-thought. When your documents have headings, sections, or list items, use those as parent boundaries for coherent context.

7. **The trade-offs are manageable.** Parent-child adds 1.5–2x storage overhead and a docstore lookup at query time. For most production systems, this is negligible compared to the +10–20% gain in answer faithfulness that typically results.

---

*Up next: Lesson 8.5 — Knowledge graphs and GraphRAG: what happens when your data has relationships that vector search alone cannot capture? When facts are connected (entity A owns entity B which employs entity C), embedding similarity finds relevant nodes but cannot follow the graph edges. GraphRAG adds that traversal capability.*